# L13b: Synthetic Markets of Many Stocks
Previously, we built a jump hidden Markov model of the daily growth rate of one asset. How do we simulate many stocks that move together, as real stocks in the same market and sector do?

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Test the single-asset model out of sample:__ Compare paths simulated from the jump hidden Markov model of `SPY` with the daily growth rates that came after its fitting period. Turn the jump episodes off and compare both models, including their volatility clustering, on the same data.
> * __Compose a synthetic market of many stocks:__ Combine each stock's baseline with market, sector, and stock-specific residual components to build synthetic stocks that move together like real ones. Describe the jump hidden Markov models that supply the market, sector, and residual draws.
> * __Size the residual component and sample the universe:__ Choose the scale of each stock's residual component so that the sector and residual components keep the variance left after the market fit. Draw one synthetic path of every stock, holding each stock's market sensitivity fixed or letting it drift.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
>
> * An out-of-sample test compares a fitted model with data from after its fitting period. We test the jump hidden Markov model of `SPY`, with and without jump episodes, on the growth rates of 2025 and 2026.
> * We model a synthetic stock's growth rate as $g_{i} = \alpha_{i} + \beta_{i}\,g_{m} + \delta_{i}\,f_{s} + \psi_{i}\,\tilde{\varepsilon}_{i}$, with a stock-specific baseline $\alpha_{i}$. Shared market and sector components capture correlations between stocks. The residual draw $\tilde{\varepsilon}_{i}$ is centered on zero and represents the movement left after subtracting the baseline, market, and sector contributions.
> * The residual scale $\psi_{i}$ targets the stock's volatility. A drifting $\beta_{i}$ lets the stock's market sensitivity change over time.
>
> __For understanding__ (you will not need to reproduce or memorize these):
>
> * The tests that compare the `SPY` model with the out-of-sample data.
> * How the sector loadings and the scale of the residual component are chosen, and the drifting $\beta$ option.
> * The sampling algorithm of the sector factor model, step by step.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L13b) lists this lecture's symbols with their meanings and units.

Let's get started!

___


## Examples

__We will work through these examples in lecture:__

> [▶ Test the `SPY` model out of sample](CHEME-5660-L13b-Example-JumpHMM-SPY-OutOfSample-Fall-2026.ipynb). In this example, we load the jump hidden Markov model of `SPY` fitted to daily growth rates from 2014 to 2024. We compare its simulated paths with the growth rates of 2025 and 2026, which the model has not seen. We then turn the jump episodes off, score both models on the same data, and compare the volatility clustering of simulated and real growth rates.

> [▶ Build a synthetic market of many stocks](CHEME-5660-L13b-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we build a universe of two stocks from each sector plus `SPY` with the sector factor model. We compare the correlations, volatilities, and market sensitivities of a synthetic path with the real data, let the sensitivities drift, and save the generator.

___


## Review: Synthetic Markets with Jump Hidden Markov Models
Previously, we built synthetic markets from jump hidden Markov models (JumpHMM). The model bins the daily growth rate into states of equal probability and estimates the transition matrix by counting moves between states.

Each simulated day, the chain takes one step. The model then draws the day's growth rate from the Student-t distribution of the new state:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-JumpHMM-Schematic.svg" width="880" style="max-width:100%; height:auto;" alt="A row of hidden states numbered 1, 2, through k, through N minus 1 and N, ordered from the most negative to the most positive daily growth rate, with double-headed arrows between neighbors. States 1 and 2 are red and braced as the most negative states S minus, and states N minus 1 and N are blue and braced as the most positive states S plus. A bell curve above each state is its emission, and state k emits mu k plus sigma k times a Student-t variable. Under state k, a note reads ordinary day: step with the estimated transition matrix. A dashed arrow from state k, marked no episode under way, leads up to a box: jump episode, starts with probability epsilon, lasts a Poisson number of days with mean lambda. From the box, a red arrow runs to the most negative states, taken on each episode day with probability p neg, and a blue arrow runs to the most positive states, taken with probability 1 minus p neg."/>
    </center>
</div>

Jump episodes start at random and last several days. On each episode day, the state comes from the most negative or the most positive states. These runs of large moves add the volatility clustering that the chain alone misses.

Let's test the jump hidden Markov model of `SPY` on data it has not seen:

> __Example:__
>
> [▶ Test the `SPY` model out of sample](CHEME-5660-L13b-Example-JumpHMM-SPY-OutOfSample-Fall-2026.ipynb). In this example, we load the jump hidden Markov model of `SPY` fitted to daily growth rates from 2014 to 2024. We compare its simulated paths with the growth rates of 2025 and 2026, which the model has not seen. We then turn the jump episodes off, score both models on the same data, and compare the volatility clustering of simulated and real growth rates.

__Today:__ We extend the jump hidden Markov model from one asset to many stocks that move together.

___


## Company Profile: AQR Capital Management
[AQR Capital Management](https://www.aqr.com/Our-Firm/About-Us) is a quantitative investment management firm. It was founded in New York City in 1998 and moved its headquarters to Greenwich, Connecticut, in 2004. The firm describes [its approach](https://www.aqr.com/What-We-Do/Our-Approach) as "systematic investing grounded in economic theory."

> __What does the firm do?__
>
> * __Betting against beta:__ In [Betting Against Beta](https://www.aqr.com/Insights/Research/Journal-Article/Betting-Against-Beta), published in 2014, AQR's Andrea Frazzini and Lasse H. Pedersen found that stocks with high beta tend to have low alpha. A portfolio that bought low-beta stocks with borrowed money and sold high-beta stocks short earned positive risk-adjusted returns.

__Explore further:__

* __Jobs and internships:__ AQR's [internships page](https://www.aqr.com/Our-Firm/Greenwich-Internships) describes a 10-week summer program in Greenwich. In November 2026, the [careers site](https://careers.aqr.com/) listed 2027 summer analyst roles in research, engineering, and business development.
* __Video:__ Watch [About AQR](https://www.youtube.com/embed/946GJrhyZtM) and [Internships at AQR](https://www.youtube.com/embed/Oow2UktGKcY) on the firm's YouTube channel.

__Connection to today's lecture:__ The betting-against-beta portfolio ranks stocks by their market sensitivity $\beta_{i}$, which sets the market component of our sector factor model. With a synthetic universe, we could test a betting-against-beta portfolio on many plausible market paths.

___


## Many Assets: The Sector Factor Model
Previously, we studied __factor models__, which describe the growth rate in terms of factors. For stock $i$ on day $t$, a general factor model is given by:

$$
g_{i}(t) = \underbrace{\alpha_{i}}_{\text{baseline}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market}} + \underbrace{\sum_{j}\delta_{ij}\,f_{j}(t)}_{\text{other factors}} + \underbrace{\varepsilon_{i}(t)}_{\text{residual}}
$$

Here $\alpha_{i}$ is the stock-specific baseline (intercept), $g_{m}(t)$ is market growth, and $f_{j}(t)$ are other factors, such as interest rates or inflation. The coefficients $\beta_{i}$ and $\delta_{ij}$ measure sensitivity to these factors. The __residual__ $\varepsilon_{i}(t)$ is the growth left after subtracting the baseline and factor contributions.

> __Definition (sector factor model):__ Each stock has one other factor besides the market, the factor $f_{s}(t)$ of its sector $s$. This factor captures co-movement left unexplained by the market.
>
> The daily growth rate of stock $i$ in sector $s$ is given by:
>
> $$
> g_{i}(t) = \underbrace{\alpha_{i}}_{\text{baseline}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market}} + \underbrace{\delta_{i}\,f_{s}(t)}_{\text{sector}} + \underbrace{\psi_{i}\,\tilde{\varepsilon}_{i}(t)}_{\text{stock residual}}
> $$
>
> The four components are:
> * __Baseline:__ The stock-specific baseline $\alpha_{i}$ is the intercept from the stock's single index fit. We hold $\alpha_{i}$ and the market sensitivity $\beta_{i}$ fixed.
> * __Market:__ The market draw $g_{m}(t)$ comes from a jump hidden Markov model of `SPY`.
> * __Sector:__ The sector draw $f_{s}(t)$ comes from a jump hidden Markov model of the average single index residual of the stocks in sector $s$. The loading $\delta_{i}$ is the slope of the stock's residual on the other residuals in its sector, scaled to match the sector's co-movement in the data.
> * __Stock-specific residual:__ The component $\psi_{i}\,\tilde{\varepsilon}_{i}(t)$ models moves left after removing the baseline, market, and sector contributions, such as an earnings surprise. We center a draw from a jump hidden Markov model of this remaining residual and size it with $\psi_{i}$.
>
> On each day, every stock shares the market draw, and every stock in a sector shares the sector draw. Each stock's residual path is sampled independently of the market, the sectors, and the other stocks' residual paths. The shared market and sector draws are the only link between stocks.

Let's write out how we construct the stock-specific residual component:

> __The stock-specific residual component (fixed β):__
>
> In the data, we subtract the market fit from the observed growth rate. We then subtract the sector contribution:
>
> $$
> \begin{aligned}
> \varepsilon_{i}(t) &= g_{i}(t)-\alpha_{i}-\beta_{i}g_{m}(t)
> && \text{residual after the market fit} \\
> u_{i}(t) &= \varepsilon_{i}(t)-\delta_{i}f_{s}(t)
> && \text{residual after market and sector}.
> \end{aligned}
> $$
>
> We fit a jump hidden Markov model to the series $u_{i}(t)$. Let $u_{i}^{\mathrm{draw}}(t)$ be a draw from that model and $\bar{u}_{i}$ its long-run mean. We center the draw on zero:
>
> $$
> \tilde{\varepsilon}_{i}(t) = u_{i}^{\mathrm{draw}}(t)-\bar{u}_{i}.
> $$
>
> The simulated sector and stock residual paths are independent. Their variances add. We use the variances in the real data to set a target for their sum:
>
> $$
> \underbrace{\delta_{i}^{2}\operatorname{Var}(f_{s})}_{\text{sector variance}}
> + \underbrace{\psi_{i}^{2}\operatorname{Var}(u_{i})}_{\text{target stock residual variance}}
> = \underbrace{\operatorname{Var}(\varepsilon_{i})}_{\text{variance left after the market fit}}.
> $$
>
> Solving for the scale gives:
>
> $$
> \boxed{\psi_{i}^{2} =
> \frac{\operatorname{Var}(\varepsilon_{i})-\delta_{i}^{2}\operatorname{Var}(f_{s})}
> {\operatorname{Var}(u_{i})}}
> \qquad
> \psi_{i} \geq 0.
> $$
>
> The sector factor includes stock $i$ in its average. The remaining series $u_{i}$ can still move with that factor in the data. The scale compensates for the covariance lost when we draw the two paths independently.
>
> The variance equation sets a target. The fitted jump hidden Markov models reproduce the data variances only approximately.

### Sampling Algorithm
The algorithm below draws one synthetic path of the whole universe over $T$ days.

__Initialize__: Given the jump hidden Markov models of the market, of each sector's factor, and of each stock's residual after its sector term; each stock's single index estimates $\alpha_{i}$ and $\beta_{i}$ and sector loading $\delta_{i}$; the scale of each stock's residual component, $\psi_{i}^{2} = \left(\operatorname{Var}(\varepsilon_{i}) - \delta_{i}^{2}\operatorname{Var}(f_{s})\right)/\operatorname{Var}(\varepsilon_{i}-\delta_{i}f_{s})$, with the variances of the real data; each stock's real starting price $S_{i}(0)$; the time step $\Delta t=1/252$ year (one trading day); and the number of days $T$. 

Draw one market path $g_{m}(1),\dotsc,g_{m}(T)$, shared by all stocks, from the market model. For each sector $s$, draw one factor path $f_{s}(1),\dotsc,f_{s}(T)$, shared by the stocks in that sector, from its sector model. Use independent random draws to generate the different paths.

For each stock $i$ in sector $s$ __do__:
1. __Residual draw:__ Draw a path from the model of the stock's remaining residual and subtract the model's long-run mean (the mean of one long simulated path) to get $\tilde{\varepsilon}_{i}(1),\dotsc,\tilde{\varepsilon}_{i}(T)$.
2. __Compose:__ For each day $t = 1,2,\dots,T$ __do__: compose the growth rate $g_{i}(t) \gets \alpha_{i} + \beta_{i}\,g_{m}(t) + \delta_{i}\,f_{s}(t) + \psi_{i}\,\tilde{\varepsilon}_{i}(t)$, and compound the price, starting from $S_{i}(0)$: $S_{i}(t) \gets S_{i}(t-1)\exp\left(g_{i}(t)\,\Delta t\right)$.

__Output__: The market path $g_{m}(t)$, and the growth rates $g_{i}(t)$ and prices $S_{i}(t)$ of every synthetic stock, for $t = 1,\dotsc,T$.

__Notes:__
1. The sector factor averages the residuals of the stocks in sector $s$, stock $i$ included. Independent draws drop the part of the stock's leftover residual that moves with the sector factor, and $\psi_{i}$ puts that part back. In a sector with many stocks, $\psi_{i}$ is close to one.
2. With the drifting β option below, $\beta_{i}$ becomes $\beta_{i}(t)$ in step 2. We also reduce $\psi_{i}$ to preserve the variance target.

### An option: a drifting β
The coefficient $\beta_{i}$ measures stock $i$'s sensitivity to market growth. Holding $\beta_{i}$ fixed assumes that stock $i$'s sensitivity to market growth rate stays constant throughout a synthetic path. Alternatively, we can let the sensitivity change over time around its single index estimate $\hat{\beta}_{i}$. 

Each day, we update it with:

$$
\beta_{i}(t) = \hat{\beta}_{i}
+ \underbrace{(1-a)\left(\beta_{i}(t-1)-\hat{\beta}_{i}\right)}_{\text{retained deviation}}
+ \underbrace{a\,k_{i}\,z_{i}(t)}_{\text{new shock}}.
$$

Here $0<a<1$ is the exponential moving average weight. A small $a$ retains most of the previous day's deviation from $\hat{\beta}_{i}$.

The shocks $z_{i}(t)$ are independent standard normal draws, with their size controlled by $k_{i}\geq 0$. In the example, we choose $k_{i}$ using the variation in β estimates updated daily from real data. These estimates also contain sampling noise.

The market contribution is now $\beta_{i}(t)g_{m}(t)$. Let's look at an example:

> __Example:__
>
> [▶ Build a synthetic market of many stocks](CHEME-5660-L13b-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we build a universe of two stocks from each sector plus `SPY` with the sector factor model. We compare the correlations, volatilities, and market sensitivities of a synthetic path with the real data, let the sensitivities drift, and save the generator.

___

## Summary
We tested the jump hidden Markov model of `SPY` on data it had not seen. We then extended the model from one asset to a universe of stocks that move together, as real stocks in the same market and sector do.

> __Key Takeaways:__
>
> * __Testing the single-asset model out of sample:__ We compared paths simulated from the jump hidden Markov model of `SPY` with the growth rates of 2025 and 2026, which came after its fitting period. We turned the jump episodes off and compared both models, including their volatility clustering, on the same data.
> * __Shared draws make synthetic stocks move together:__ We combined each stock's baseline with shared market and sector components and a stock-specific residual component. Every stock shares the market draw, and every stock in a sector shares the sector draw.
> * __Variance targets and a drifting market sensitivity:__ We sized the residual component to preserve the variance left after the market fit when combined with the sector component. As an option, we let each stock's market sensitivity drift around its single index estimate.

Synthetic markets of many stocks let us test a trading method on many plausible paths, rather than on one historical path. Knowing how the synthetic universe is built also tells us its limits: stocks in different sectors move together only through the market draw.

___


## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success.

Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___
